# SMART Baseline Training (Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/achyutmorang/wosac-sim-agents-experiments/blob/main/experiments/smart-baseline/notebooks/smart-baseline_colab.ipynb)

## Objective
- Train SMART baseline and persist checkpoints to Drive.
- Keep this notebook training-only.
- Run simulation/evaluation in dedicated eval/comparative notebooks.


In [1]:
# Step 1: Sync this repo and bootstrap deterministic Colab runtime
import os
import subprocess
import sys

REPO_URL = "https://github.com/achyutmorang/wosac-sim-agents-experiments.git"
REPO_DIR = "/content/wosac-sim-agents-experiments"

if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", "main"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", "main"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", "main", REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
for p in (REPO_DIR, os.path.join(REPO_DIR, "src")):
    if p not in sys.path:
        sys.path.insert(0, p)

from src.platform import bootstrap_colab_runtime_with_config, wosac_colab_runtime_config
runtime_cfg = wosac_colab_runtime_config(repo_url=REPO_URL, repo_branch="main", repo_dir=REPO_DIR)
bootstrap = bootstrap_colab_runtime_with_config(runtime_cfg)
print("repo_rev:", bootstrap.repo_sync.repo_rev)

if bootstrap.setup.result.get("restart_required", False):
    raise RuntimeError("Runtime restart required. Restart runtime and rerun this cell.")


[repo] existing checkout: /content/wosac-sim-agents-experiments
[drive] /content/drive already mounted
[drive] Verified read/write access: /content/drive/MyDrive/wosac_experiments
[setup] Starting deterministic environment bootstrap
[setup] $ /usr/bin/python3 -c import jax, waymax, numpy as np, pandas as pd, scipy, sklearn; from numpy._core.umath import _center, _expandtabs; print('ok', np.__version__, pd.__version__, scipy.__version__, sklearn.__version__, jax.__version__)
[setup] Core runtime already healthy; skipping dependency install (ok 2.2.6 2.2.3 1.14.1 1.6.1 0.7.2).
[setup] $ /usr/bin/python3 -c import numpy as np; from numpy._core.umath import _center, _expandtabs; print(np.__version__, np.__file__)
[setup] NumPy probe passed (2.2.6 /usr/local/lib/python3.12/dist-packages/numpy/__init__.py).
[setup] $ /usr/bin/python3 -c import jax, waymax, numpy as np, pandas as pd, scipy, sklearn; from numpy._core.umath import _center, _expandtabs; print('ok', np.__version__, pd.__version__

In [3]:
%env WOSAC_RUN_MODE=pilot
%env SMART_BASELINE_PROFILE=smoke
%env SMART_RUN_PREPROCESS=0
%env SMART_RUN_TRAIN=1


env: WOSAC_RUN_MODE=pilot
env: SMART_BASELINE_PROFILE=smoke
env: SMART_RUN_PREPROCESS=0
env: SMART_RUN_TRAIN=1


In [4]:
# Step 2: Load config, select run mode, stage data from GCS, and build preprocess policy
import hashlib
import json
import os
import subprocess
import sys
from datetime import datetime, timezone
from pathlib import Path

from src.workflows import bootstrap_experiment_pack, load_experiment_config


def _bool_env(name: str, default: bool = False) -> bool:
    text = os.environ.get(name, "").strip().lower()
    if text in {"1", "true", "yes", "y", "on"}:
        return True
    if text in {"0", "false", "no", "n", "off"}:
        return False
    return bool(default)


def _ensure_colab_auth() -> None:
    if 'google.colab' not in sys.modules:
        return
    try:
        from google.colab import auth  # type: ignore

        auth.authenticate_user()
        print('colab_gcs_auth: OK')
    except Exception as exc:
        print(f'colab_gcs_auth: skipped ({exc})')


def _gcloud_ls(pattern: str) -> list[str]:
    try:
        out = subprocess.check_output(
            ["bash", "-lc", f"gcloud storage ls '{pattern}'"],
            text=True,
            stderr=subprocess.STDOUT,
        )
        rows = [line.strip() for line in out.splitlines() if line.strip()]
        return rows
    except Exception:
        return []


def _choose_shards(pattern: str, limit: int) -> list[str]:
    rows = sorted(_gcloud_ls(pattern))
    if limit > 0:
        return rows[:limit]
    return rows


def _local_shards(split_dir: Path) -> list[Path]:
    return sorted([p for p in split_dir.glob('*.tfrecord*') if p.is_file()])


def _count_processed(split_dir: Path) -> int:
    files = [p for p in split_dir.glob('*.pkl') if p.is_file()]
    files += [p for p in split_dir.glob('*.pickle') if p.is_file()]
    return len(files)


def _stage_split(*, gcs_root: str, split_name: str, shard_limit: int, local_split_dir: Path, force: bool) -> dict:
    local_split_dir.mkdir(parents=True, exist_ok=True)
    pattern = f"{gcs_root}/{split_name}/{split_name}.tfrecord-*"
    remote_shards = _choose_shards(pattern, shard_limit)
    existing = {p.name for p in _local_shards(local_split_dir)}
    pending = remote_shards if force else [u for u in remote_shards if Path(u).name not in existing]

    copied = 0
    errors: list[str] = []
    for uri in pending:
        try:
            subprocess.run(
                ["gcloud", "storage", "cp", "--no-clobber", uri, str(local_split_dir) + "/"],
                check=True,
            )
            copied += 1
        except Exception as exc:
            errors.append(f"{uri}: {type(exc).__name__}")

    local_files = _local_shards(local_split_dir)
    return {
        "split": split_name,
        "pattern": pattern,
        "requested_limit": int(shard_limit),
        "remote_candidates": len(remote_shards),
        "pending_before_copy": len(pending),
        "copied": int(copied),
        "local_count": len(local_files),
        "local_sample": [str(p) for p in local_files[:5]],
        "errors": errors,
    }


EXPERIMENT_SLUG = "smart-baseline"
bundle = bootstrap_experiment_pack(slug=EXPERIMENT_SLUG, repo_root=".")
cfg = load_experiment_config(slug=EXPERIMENT_SLUG, repo_root=".")
display(bundle.summary_table)

RUN = dict(cfg.get("run", {}))
SMART = dict(cfg.get("smart", {}))
SMART_PROFILES = dict(SMART.get("profiles", {}))

RUN_NAME = str(RUN.get("run_name", "dev"))
RUN_PREFIX = str(RUN.get("run_prefix", "smart_baseline"))
PERSIST_ROOT = str(RUN.get("persist_root", "/content/drive/MyDrive/wosac_experiments"))
RESUME_FROM_EXISTING = bool(RUN.get("resume_from_existing", True))
RUN_TAG = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
cfg_hash = hashlib.sha256(json.dumps(cfg, sort_keys=True).encode("utf-8")).hexdigest()
persist_run_dir = Path(PERSIST_ROOT) / f"{RUN_PREFIX}_{RUN_NAME}"
persist_run_dir.mkdir(parents=True, exist_ok=True)
RUN_OUTPUT_DIR = persist_run_dir / "outputs" / RUN_TAG
RUN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_MODE = os.environ.get("WOSAC_RUN_MODE", "pilot").strip().lower() or "pilot"
if RUN_MODE not in {"pilot", "full"}:
    print(f"Unknown WOSAC_RUN_MODE={RUN_MODE!r}; falling back to 'pilot'.")
    RUN_MODE = "pilot"

default_profile = "smoke" if RUN_MODE == "pilot" else str(SMART.get("profile", "paper_repro"))
SMART_PROFILE = os.environ.get("SMART_BASELINE_PROFILE", default_profile).strip() or default_profile
SMART_EFFECTIVE = dict(SMART)
if SMART_PROFILE in SMART_PROFILES:
    SMART_EFFECTIVE.update(dict(SMART_PROFILES[SMART_PROFILE]))

raw_root_override = os.environ.get("SMART_RAW_DATA_ROOT", "").strip()
processed_root_override = os.environ.get("SMART_PROCESSED_DATA_ROOT", "").strip()
if raw_root_override:
    SMART_EFFECTIVE["raw_data_root"] = raw_root_override
if processed_root_override:
    SMART_EFFECTIVE["processed_data_root"] = processed_root_override

BASELINE_CKPT_DIR = os.environ.get("SMART_BASELINE_CKPT_DIR", "").strip() or str(persist_run_dir / "checkpoints" / f"smart_baseline_{SMART_PROFILE}")
SMART_RESUME_CKPT = os.environ.get("SMART_RESUME_CKPT", "").strip()
SMART_TRAIN_SEED = int(os.environ.get("SMART_TRAIN_SEED", str(SMART_EFFECTIVE.get("seed", 2))))

DATA_SOURCE_MODE = os.environ.get("SMART_DATA_SOURCE", "gcs_stage").strip().lower() or "gcs_stage"
RUN_DATA_STAGE = _bool_env("SMART_RUN_DATA_STAGE", True)
FORCE_DATA_REDOWNLOAD = _bool_env("SMART_FORCE_DATA_REDOWNLOAD", False)
GCS_DATASET_ROOT = os.environ.get("SMART_GCS_DATASET_ROOT", "gs://waymo_open_dataset_motion_v_1_3_1/uncompressed/scenario").strip().rstrip("/")
TRAIN_SPLIT = os.environ.get("SMART_GCS_TRAIN_SPLIT", "training").strip() or "training"
VAL_SPLIT = os.environ.get("SMART_GCS_VAL_SPLIT", "validation").strip() or "validation"
TRAIN_SHARD_LIMIT = int(os.environ.get("SMART_GCS_TRAIN_SHARDS", "8" if RUN_MODE == "pilot" else "64"))
VAL_SHARD_LIMIT = int(os.environ.get("SMART_GCS_VAL_SHARDS", "2" if RUN_MODE == "pilot" else "8"))

RAW_ROOT = Path(str(SMART_EFFECTIVE.get("raw_data_root", "/content/SMART/data/waymo/scenario"))).expanduser()
PROCESSED_ROOT = Path(str(SMART_EFFECTIVE.get("processed_data_root", "/content/SMART/data/waymo_processed"))).expanduser()
RAW_TRAIN_DIR = RAW_ROOT / "training"
RAW_VAL_DIR = RAW_ROOT / "validation"
PROCESSED_TRAIN_DIR = PROCESSED_ROOT / "training"
PROCESSED_VAL_DIR = PROCESSED_ROOT / "validation"

for d in [RAW_TRAIN_DIR, RAW_VAL_DIR, PROCESSED_TRAIN_DIR, PROCESSED_VAL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

data_stage_manifest = {
    "data_source_mode": DATA_SOURCE_MODE,
    "run_data_stage": bool(RUN_DATA_STAGE),
    "force_data_redownload": bool(FORCE_DATA_REDOWNLOAD),
    "gcs_dataset_root": GCS_DATASET_ROOT,
    "train_split": TRAIN_SPLIT,
    "val_split": VAL_SPLIT,
    "train_shard_limit": int(TRAIN_SHARD_LIMIT),
    "val_shard_limit": int(VAL_SHARD_LIMIT),
    "results": {},
}

if DATA_SOURCE_MODE == "gcs_stage" and RUN_DATA_STAGE:
    _ensure_colab_auth()
    data_stage_manifest["results"]["training"] = _stage_split(
        gcs_root=GCS_DATASET_ROOT,
        split_name=TRAIN_SPLIT,
        shard_limit=TRAIN_SHARD_LIMIT,
        local_split_dir=RAW_TRAIN_DIR,
        force=FORCE_DATA_REDOWNLOAD,
    )
    data_stage_manifest["results"]["validation"] = _stage_split(
        gcs_root=GCS_DATASET_ROOT,
        split_name=VAL_SPLIT,
        shard_limit=VAL_SHARD_LIMIT,
        local_split_dir=RAW_VAL_DIR,
        force=FORCE_DATA_REDOWNLOAD,
    )
else:
    data_stage_manifest["results"]["note"] = "staging_skipped"

raw_train_count = len(_local_shards(RAW_TRAIN_DIR))
raw_val_count = len(_local_shards(RAW_VAL_DIR))
processed_train_count = _count_processed(PROCESSED_TRAIN_DIR)
processed_val_count = _count_processed(PROCESSED_VAL_DIR)

SMART_FORCE_PREPROCESS = _bool_env("SMART_FORCE_PREPROCESS", False)
DEFAULT_RUN_PREPROCESS = bool(SMART_FORCE_PREPROCESS or processed_train_count == 0 or processed_val_count == 0)

preprocess_policy = {
    "force_preprocess": bool(SMART_FORCE_PREPROCESS),
    "default_run_preprocess": bool(DEFAULT_RUN_PREPROCESS),
    "processed_train_count": int(processed_train_count),
    "processed_val_count": int(processed_val_count),
    "raw_train_count": int(raw_train_count),
    "raw_val_count": int(raw_val_count),
}

print("RUN_TAG:", RUN_TAG)
print("persist_run_dir:", persist_run_dir)
print("RUN_OUTPUT_DIR:", RUN_OUTPUT_DIR)
print("RUN_MODE:", RUN_MODE)
print("SMART_PROFILE:", SMART_PROFILE)
print("BASELINE_CKPT_DIR:", BASELINE_CKPT_DIR)
print("RESUME_FROM_EXISTING:", RESUME_FROM_EXISTING)
print("SMART_RESUME_CKPT (optional explicit):", SMART_RESUME_CKPT or "<auto>")
print("SMART repo commit target:", SMART_EFFECTIVE.get("repo_commit", ""))
print("SMART train config:", SMART_EFFECTIVE.get("train_config", ""))
print("SMART val config:", SMART_EFFECTIVE.get("val_config", ""))
print("SMART train seed:", SMART_TRAIN_SEED)
print("SMART raw_data_root:", SMART_EFFECTIVE.get("raw_data_root", ""))
print("SMART processed_data_root:", SMART_EFFECTIVE.get("processed_data_root", ""))
print("cfg_hash:", cfg_hash)
print("data_stage_manifest:")
print(json.dumps(data_stage_manifest, indent=2, sort_keys=True))
print("preprocess_policy:")
print(json.dumps(preprocess_policy, indent=2, sort_keys=True))



,field,value
0,slug,smart-baseline
1,title,SMART Baseline Wrapper
2,objective,Reproduce SMART baseline with thin wrapper flo...
3,notebook,experiments/smart-baseline/notebooks/smart-bas...
4,workflow,src/workflows/smart_baseline_flow.py
5,config_file,/content/wosac-sim-agents-experiments/configs/...


colab_gcs_auth: OK
RUN_TAG: 20260330T171856Z
persist_run_dir: /content/drive/MyDrive/wosac_experiments/smart_baseline_dev
RUN_OUTPUT_DIR: /content/drive/MyDrive/wosac_experiments/smart_baseline_dev/outputs/20260330T171856Z
RUN_MODE: pilot
SMART_PROFILE: smoke
BASELINE_CKPT_DIR: /content/drive/MyDrive/wosac_experiments/smart_baseline_dev/checkpoints/smart_baseline_smoke
RESUME_FROM_EXISTING: True
SMART_RESUME_CKPT (optional explicit): <auto>
SMART repo commit target: 42e658542b03dd14e1c36c63e214994e4b8c7b36
SMART train config: configs/train/train_scalable.yaml
SMART val config: configs/validation/validation_scalable.yaml
SMART train seed: 2
SMART raw_data_root: /content/SMART/data/waymo/scenario
SMART processed_data_root: /content/SMART/data/waymo_processed
cfg_hash: 3a55f8daa9f07df7b3389f7127e4ee88bab0f11075d1ac8e306b7777e205820f
data_stage_manifest:
{
  "data_source_mode": "gcs_stage",
  "force_data_redownload": false,
  "gcs_dataset_root": "gs://waymo_open_dataset_motion_v_1_3_1/unco

In [5]:
# Step 3: Build SMART baseline training command plan (train-only)
from src.workflows import run_smart_baseline_flow

flow_bundle = run_smart_baseline_flow(
    run_tag=RUN_TAG,
    run_name=RUN_NAME,
    run_prefix=RUN_PREFIX,
    persist_root=PERSIST_ROOT,
    repo_root=".",
    resume_from_existing=RESUME_FROM_EXISTING,
    smart_repo_url=str(SMART_EFFECTIVE.get("repo_url", "https://github.com/rainmaker22/SMART.git")),
    smart_repo_branch=str(SMART_EFFECTIVE.get("branch", "main")),
    smart_repo_commit=str(SMART_EFFECTIVE.get("repo_commit", "")).strip(),
    smart_repo_dir=str(SMART_EFFECTIVE.get("repo_dir", "/content/SMART")),
    smart_train_config=str(SMART_EFFECTIVE.get("train_config", "configs/train/train_scalable.yaml")),
    smart_val_config=str(SMART_EFFECTIVE.get("val_config", "configs/validation/validation_scalable.yaml")),
    smart_ckpt_path=SMART_RESUME_CKPT,
    smart_save_ckpt_path=BASELINE_CKPT_DIR,
    smart_raw_data_root=str(SMART_EFFECTIVE.get("raw_data_root", "/content/SMART/data/waymo/scenario")),
    smart_processed_data_root=str(SMART_EFFECTIVE.get("processed_data_root", "/content/SMART/data/waymo_processed")),
    smart_install_pyg=bool(SMART_EFFECTIVE.get("install_pyg", True)),
    smart_env_lockfile=str(SMART_EFFECTIVE.get("env_lockfile", "")).strip(),
    smart_train_seed=SMART_TRAIN_SEED,
    smart_deterministic_train=bool(SMART_EFFECTIVE.get("deterministic_train", True)),
    smart_train_launcher_path=str(SMART_EFFECTIVE.get("train_launcher_path", "scripts/smart_train_repro.py")),
    smart_profile=SMART_PROFILE,
    smart_force_preprocess=bool(preprocess_policy.get("force_preprocess", False)),
    sync_smart_repo=True,
)

print("flow_summary:")
print(json.dumps(flow_bundle.summary, indent=2, sort_keys=True))

if flow_bundle.summary.get("status") == "sync_failed":
    raise RuntimeError(
        f"SMART repo sync failed before command execution: {flow_bundle.summary.get('sync_error', 'unknown')}"
    )
print("resolved_resume_checkpoint:", flow_bundle.summary.get("smart_ckpt_path", "") or "<none>")
print("train_cmd:")
print(flow_bundle.command_plan["train_cmd"])
print("validate_cmd (not used in this notebook):")
print(flow_bundle.command_plan["validate_cmd"])


flow_summary:
{
  "checkpoint_manifest": {
    "checkpoints": [],
    "pretrain_ckpt_path": "",
    "pretrain_ckpt_sha256": "",
    "resume": {
      "candidate_sample": [],
      "checkpoint_exists": false,
      "checkpoint_path": "",
      "num_candidates": 0,
      "source": "none"
    },
    "save_ckpt_exists": false,
    "save_ckpt_path": "/content/drive/MyDrive/wosac_experiments/smart_baseline_dev/checkpoints/smart_baseline_smoke"
  },
  "config_hash": "da907ba8ef9cbeb9bbf881ec8797dbb103166bc309b54c814a7e27b0c9316715",
  "created_utc": "2026-03-30T17:19:20Z",
  "data_manifest": {
    "processed_data_root": "/content/SMART/data/waymo_processed",
    "raw_data_root": "/content/SMART/data/waymo/scenario",
    "splits": {
      "training": {
        "processed": {
          "exists": true,
          "listing_sha256": "",
          "num_files": 0,
          "path": "/content/SMART/data/waymo_processed/training",
          "sample_files": [],
          "total_bytes": 0
        },
    

In [6]:
# Step 4: Optional training execution (no evaluation in this notebook)
import os
import json
import subprocess

RUN_SETUP = _bool_env("SMART_RUN_SETUP", False)
RUN_PREPROCESS = _bool_env("SMART_RUN_PREPROCESS", bool(preprocess_policy.get("default_run_preprocess", False)))
RUN_TRAIN = _bool_env("SMART_RUN_TRAIN", False)
AUTO_SETUP = _bool_env("SMART_AUTO_SETUP", True)

if bool(preprocess_policy.get("force_preprocess", False)):
    RUN_PREPROCESS = True

if RUN_TRAIN and not RUN_SETUP and AUTO_SETUP:
    RUN_SETUP = True
    print("SMART_AUTO_SETUP=1 -> enabling setup before train.")

if RUN_PREPROCESS:
    if int(preprocess_policy.get("raw_train_count", 0)) <= 0 or int(preprocess_policy.get("raw_val_count", 0)) <= 0:
        raise RuntimeError(
            "RUN_PREPROCESS requested but raw shards are missing. "
            "Rerun Step 2 with SMART_RUN_DATA_STAGE=1 or point SMART_RAW_DATA_ROOT to existing local shards."
        )
    for _d in [PROCESSED_TRAIN_DIR, PROCESSED_VAL_DIR]:
        _d.mkdir(parents=True, exist_ok=True)

stage_flags = {
    "run_setup": bool(RUN_SETUP),
    "run_preprocess": bool(RUN_PREPROCESS),
    "run_train": bool(RUN_TRAIN),
    "auto_setup": bool(AUTO_SETUP),
    "run_mode": RUN_MODE,
}
print("stage_flags:")
print(json.dumps(stage_flags, indent=2, sort_keys=True))


def _run_cmd(cmd: str, idx: int, total: int) -> None:
    print(f"[exec {idx}/{total}] {cmd}")
    merged_cmd = f"export PYTHONUNBUFFERED=1; {cmd}"
    process = subprocess.Popen(
        ["bash", "-lc", merged_cmd],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    recent_lines = []
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
        recent_lines.append(line)
        if len(recent_lines) > 80:
            recent_lines.pop(0)
    return_code = int(process.wait())
    if return_code == 0:
        return
    smart_repo_dir = str(SMART_EFFECTIVE.get("repo_dir", "/content/SMART"))
    diag = {
        "failed_command": cmd,
        "exit_code": return_code,
        "smart_repo_dir": smart_repo_dir,
        "smart_repo_exists": bool(os.path.isdir(smart_repo_dir)),
        "data_preprocess_exists": bool(os.path.isfile(os.path.join(smart_repo_dir, "data_preprocess.py"))),
        "raw_train_dir": str(RAW_TRAIN_DIR),
        "raw_val_dir": str(RAW_VAL_DIR),
        "raw_train_count": int(preprocess_policy.get("raw_train_count", 0)),
        "raw_val_count": int(preprocess_policy.get("raw_val_count", 0)),
        "processed_train_dir": str(PROCESSED_TRAIN_DIR),
        "processed_val_dir": str(PROCESSED_VAL_DIR),
        "processed_train_count": int(preprocess_policy.get("processed_train_count", 0)),
        "processed_val_count": int(preprocess_policy.get("processed_val_count", 0)),
        "recent_output": "".join(recent_lines[-20:]),
    }
    print("step4_diagnostics:")
    print(json.dumps(diag, indent=2, sort_keys=True))
    raise subprocess.CalledProcessError(return_code, ["bash", "-lc", merged_cmd])


cmds = []
if RUN_SETUP:
    cmds.append(flow_bundle.command_plan["setup_cmd"])
if RUN_PREPROCESS:
    cmds.extend([
        flow_bundle.command_plan["preprocess_train_cmd"],
        flow_bundle.command_plan["preprocess_val_cmd"],
    ])
if RUN_TRAIN:
    cmds.append(flow_bundle.command_plan["train_cmd"])

for i, cmd in enumerate(cmds, start=1):
    _run_cmd(cmd, i, len(cmds))

print("Training execution block done.")


SMART_AUTO_SETUP=1 -> enabling setup before train.
stage_flags:
{
  "auto_setup": true,
  "run_mode": "pilot",
  "run_preprocess": false,
  "run_setup": true,
  "run_train": true
}
[exec 1/2] cd /content/wosac-sim-agents-experiments && python /content/wosac-sim-agents-experiments/scripts/ensure_smart_train_runtime.py --smart-repo-dir /content/SMART
[smart-train-setup] missing pytorch_lightning: ModuleNotFoundError: No module named 'pytorch_lightning'
[smart-train-setup] $ /usr/bin/python3 -m pip install --upgrade pytorch-lightning>=2.4,<2.6
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 831.6/831.6 kB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 65.1 MB/s eta 0:00:00
[smart-train-setup] missing waymo_open_dataset: ModuleNotFoundError: No module named 'waymo_open_dataset'
[smart-train-setup] $ /usr/bin/python3 -m pip install --upgrade --no-deps waymo-open-dataset-tf-2-12-0==1.6.7
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 102.4 MB/s eta 0:00:

In [7]:
# Step 5: Write baseline training artifact + run manifest (Drive)
from pathlib import Path

from src.workflows import build_training_run_manifest, write_json

stage_flags = globals().get("stage_flags", {"run_setup": False, "run_preprocess": False, "run_train": False, "run_mode": RUN_MODE})
ckpt_files = sorted([str(p) for p in Path(BASELINE_CKPT_DIR).expanduser().glob("*.ckpt")])
resume_resolution = dict(flow_bundle.summary.get("resume_resolution", {}) or {})

payload = {
    "run_id": "smart_baseline_train_v0",
    "status": str(flow_bundle.summary.get("status", "unknown")),
    "run_tag": RUN_TAG,
    "run_mode": RUN_MODE,
    "smart_profile": SMART_PROFILE,
    "checkpoint_dir": BASELINE_CKPT_DIR,
    "checkpoint_files": ckpt_files,
    "resume_resolution": resume_resolution,
    "run_output_dir": str(RUN_OUTPUT_DIR),
    "stage_flags": stage_flags,
    "data_stage_manifest": data_stage_manifest,
    "preprocess_policy": preprocess_policy,
    "flow_summary": flow_bundle.summary,
    "command_plan": flow_bundle.command_plan,
    "flow_artifacts": flow_bundle.artifacts,
    "notes": [
        "Training-only notebook. Use smart_rollout_simulation_colab.ipynb and smart_checkpoint_eval_colab.ipynb for simulation/evaluation.",
    ],
    "provenance": {
        "config_hash": cfg_hash,
        "created_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "experiment_slug": EXPERIMENT_SLUG,
    },
}

drive_path = RUN_OUTPUT_DIR / "smart_baseline_train_v0.json"
drive_path.parent.mkdir(parents=True, exist_ok=True)
drive_path.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")

run_manifest = build_training_run_manifest(
    run_id="smart_baseline_train_v0",
    run_tag=RUN_TAG,
    experiment_slug=EXPERIMENT_SLUG,
    run_name=RUN_NAME,
    run_prefix=RUN_PREFIX,
    persist_root=PERSIST_ROOT,
    repo_root=".",
    config_hash=cfg_hash,
    flow_summary=flow_bundle.summary,
    stage_flags=stage_flags,
    checkpoint_dir=BASELINE_CKPT_DIR,
    resume_checkpoint_path=str(flow_bundle.summary.get("smart_ckpt_path", "")),
    resume_checkpoint_source=str(resume_resolution.get("source", "")),
    extra={
        "flow_artifacts": flow_bundle.artifacts,
        "checkpoint_files": ckpt_files,
        "data_stage_manifest": data_stage_manifest,
        "preprocess_policy": preprocess_policy,
    },
)
manifest_path = RUN_OUTPUT_DIR / "run_manifest.json"
write_json(manifest_path, run_manifest)

print("drive_path:", drive_path)
print("manifest_path:", manifest_path)
print("num_ckpts_found:", len(ckpt_files))


drive_path: /content/drive/MyDrive/wosac_experiments/smart_baseline_dev/outputs/20260330T171856Z/smart_baseline_train_v0.json
manifest_path: /content/drive/MyDrive/wosac_experiments/smart_baseline_dev/outputs/20260330T171856Z/run_manifest.json
num_ckpts_found: 5
